## Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.



## Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

GOOGLE_API_KEY = "GOOGLE_API_KEY"
GROQ_API_KEY = "GROQ_API_KEY"
OPENAI_API_KEY = "OPENAI_API_KEY"

def require_env(name: str) -> str:
    value = os.getenv(name)
    if value is None:
        raise ValueError(f"{name} is not set. Add it to your .env file.")
    os.environ[name] = value
    return value

google_api_key = require_env(GOOGLE_API_KEY)
groq_api_key = require_env(GROQ_API_KEY)
openai_api_key = require_env(OPENAI_API_KEY)

In [9]:
from langchain.chat_models import init_chat_model
model = init_chat_model("groq:qwen/qwen3.8-27b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, client=<groq.resources.chat.completions.Completions object at 0x71792b9fa520>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x71792b9f9ba0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [11]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [12]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, client=<groq.resources.chat.completions.Completions object at 0x71792b9fa520>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x71792b9f9ba0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function': {'name': 'Movie', 'de

In [10]:
model.invoke("Provide details about the moview Inception")

AIMessage(content='**Inception** is a 2010 science fiction action film written, produced, and directed by **Christopher Nolan**. It is widely regarded as one of the most complex and influential blockbusters of the 21st century.\n\nHere are the key details:\n\n### 🎬 Basic Information\n- **Release Date**: July 16, 2010 (USA)\n- **Runtime**: 148 minutes\n- **Genre**: Science fiction, action, thriller\n- **Studio**: Warner Bros. Pictures\n- **Director/Writer/Producer**: Christopher Nolan\n- **Music**: Hans Zimmer\n\n### 🎭 Main Cast\n- **Leonardo DiCaprio** as **Cobb** (the protagonist)\n- **Joseph Gordon-Levitt** as **Arthur** (Cobb’s point man)\n- **Elliot Page** as **Ariadne** (an architect who designs dream spaces)\n- **Tom Hardy** as **Eames** (a forger who creates disguises in dreams)\n- **Ken Watanabe** as **Saito** (the client who hires Cobb)\n- **Marion Cotillard** as **Mal** (Cobb’s wife, who appears in his subconscious)\n- **Michael Caine** as **Miles** (a professor and Cobb’s me

In [13]:
model_with_structure.invoke("Provide details about the moview Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

Το include_raw=True στο model.with_structured_output(Movie, include_raw=True) αλλάζει τη μορφή του output που επιστρέφει το LangChain.

Χωρίς include_raw (default False):
Το invoke() επιστρέφει απευθείας ένα instance του Pydantic model (π.χ. Movie(title=..., year=..., ...)). Αν το parsing αποτύχει (π.χ. το LLM επέστρεψε κάτι που δεν ταιριάζει στο schema), παίρνεις exception.

Με include_raw=True:
Το invoke() επιστρέφει ένα dict με τρία keys:
{
    "raw": AIMessage(...),      # το ακατέργαστο μήνυμα του μοντέλου (tool call, usage metadata, κλπ.)
    "parsed": Movie(...),        # το parsed Pydantic object (ή None αν απέτυχε)
    "parsing_error": None         # το exception αν το parsing απέτυχε, αλλιώς None
}



In [14]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'r07y66nsk', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 355, 'total_tokens': 423, 'completion_time': 0.180261565, 'completion_tokens_details': None, 'prompt_time': 0.025819315, 'prompt_tokens_details': None, 'queue_time': 0.033195505, 'total_time': 0.20608088}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_8999620a6e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0ee2a-89cb-7a21-8b87-9e372e96310f-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.8, 'title': 'Inception', 'year': 2010}, 'id': 'r07y66nsk', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 355, 'output_tokens': 68, 'total_

### Nested Structure


In [15]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal Cobb'), Actor(name='Michael Caine', role='Miles')], genres=['Science Fiction', 'Action', 'Thriller'], budget=160.0)

# TypedDict

TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [19]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_typedict=model.with_structured_output(MovieDict)
response=model_with_typedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [20]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man / Tony Stark'},
  {'name': 'Chris Evans', 'role': 'Captain America / Steve Rogers'},
  {'name': 'Scarlett Johansson', 'role': 'Black Widow / Natasha Romanoff'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Mark Ruffalo', 'role': 'Hulk / Bruce Banner'},
  {'name': 'Jeremy Renner', 'role': 'Hawkeye / Clint Barton'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'}],
 'genres': ['Action', 'Adventure', 'Science Fiction'],
 'title': 'Avengers',
 'year': 2012}

## BaseModel vs TypedDict

Και οι δύο χρησιμοποιούνται για να ορίσεις το schema στο with_structured_output(), αλλά διαφέρουν σε τι είναι και τι προσφέρουν.

TypedDict (από το typing)
from typing_extensions import TypedDict

class Movie(TypedDict):
    title: str
    year: int
    director: str
    rating: float
- Είναι απλά ένα type hint — δεν υπάρχει καμία runtime validation. Είναι απλό dict στην πραγματικότητα.
- Το αποτέλεσμα του invoke() είναι ένα κανονικό Python dict, όχι object με attributes (δηλαδή response["title"], όχι response.title).
- Δεν κάνει coercion/validation τιμών (π.χ. αν το LLM επιστρέψει year ως string "2010" αντί για int, δεν θα το πιάσει/διορθώσει).
- Πιο ελαφρύ, χωρίς dependency στο Pydantic.
- Υποστηρίζει Annotated για descriptions:
title: Annotated[str, ..., "The title of the movie"]

BaseModel (από το Pydantic)
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
- Κάνει runtime validation — αν τα δεδομένα δεν ταιριάζουν στο schema (λάθος τύπος, λείπει field), πετάει ValidationError.
- Κάνει type coercion όπου είναι δυνατό (π.χ. "2010" → 2010 αν το field είναι int).
- Το αποτέλεσμα είναι ένα πλήρες object με attribute access (response.title), methods όπως .model_dump(), .model_dump_json(), κλπ.
- Υποστηρίζει validators, defaults, nested models, πιο πλούσιο ecosystem.
- Λίγο πιο "βαρύ" αλλά πολύ πιο ασφαλές για production.

Πρακτική διαφορά στο LangChain:
- with_structured_output(MovieTypedDict) → επιστρέφει dict
- with_structured_output(MoviePydantic) → επιστρέφει Pydantic instance με validation

Πότε ποιο:
- TypedDict: γρήγορο prototyping, δεν σε νοιάζει η validation, θες απλό dict output.
- BaseModel: production code, θες σιγουριά ότι τα δεδομένα είναι σωστού τύπου, θες extra methods/validators.

# DataClasses

Τα dataclasses είναι ένα built-in χαρακτηριστικό της Python (από το module dataclasses) που σου επιτρέπει να φτιάξεις γρήγορα classes που κυρίως κρατάνε δεδομένα, χωρίς να γράφεις boilerplate κώδικα (όπως __init__, __repr__, __eq__).

from dataclasses import dataclass, field

@dataclass
class Movie:
    title: str
    year: int
    director: str
    rating: float = 0.0  # default value


Το decorator @dataclass αυτόματα δημιουργεί:
- __init__ — μπορείς να κάνεις Movie(title="Inception", year=2010, director="Nolan", rating=8.8)
- __repr__ — ωραίο print output: Movie(title='Inception', year=2010, ...)
- __eq__ — σύγκριση βάσει τιμών (Movie(...) == Movie(...))

Πώς διαφέρει από BaseModel/TypedDict (στο context του LangChain structured output):

```table
┌────────────────────┬────────────────────────────┬──────────────────────────┬─────────────────────────────────────────┐
│                    │         dataclass          │        TypedDict         │                BaseModel                │
├────────────────────┼────────────────────────────┼──────────────────────────┼─────────────────────────────────────────┤
│ Runtime validation │ ❌ Όχι                      │ ❌ Όχι                    │ ✅ Ναι                                   │
├────────────────────┼────────────────────────────┼──────────────────────────┼─────────────────────────────────────────┤
│ Type coercion      │ ❌ Όχι                      │ ❌ Όχι                    │ ✅ Ναι                                   │
├────────────────────┼────────────────────────────┼──────────────────────────┼─────────────────────────────────────────┤
│ Access             │ response.title (attribute) │ response["title"] (dict) │ response.title (attribute)              │
├────────────────────┼────────────────────────────┼──────────────────────────┼─────────────────────────────────────────┤
│ Εξάρτηση           │ Built-in Python            │ Built-in typing          │ Pydantic (external lib)                 │
├────────────────────┼────────────────────────────┼──────────────────────────┼─────────────────────────────────────────┤
│ Extra methods      │ Λίγα                       │ Κανένα                   │ Πολλά (.model_dump(), validators, κλπ.) │
└────────────────────┴────────────────────────────┴──────────────────────────┴─────────────────────────────────────────┘
```

Στο LangChain, το with_structured_output() υποστηρίζει και τα τρία ως schema definitions. Η dataclass είναι μια μεσαία λύση: σου δίνει attribute-style access (σαν BaseModel) χωρίς όμως validation/coercion (σαν TypedDict) — απλά είναι ένα ελαφρύ, native Python container για δομημένα δεδομένα, χρήσιμο όταν δεν χρειάζεσαι validation αλλά θες πιο "καθαρό" object από dict.


In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-5-nano",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result